# Product Return Prediction using Predictive Analytics
## Academic Project: Complete Machine Learning Pipeline

**Objective:** Predict whether an e-commerce order is likely to be returned ($1$) or retained ($0$) using **Logistic Regression** and a scikit-learn preprocessing pipeline.

### Pipeline Steps:
1. Data Collection & Inspection
2. Data Hygiene & Missing Value Analysis
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Stratified Train-Test Split (80/20)
6. Preprocessing via ColumnTransformer
7. Baseline Model Comparison (Dummy Classifier)
8. 5-Fold Stratified Cross-Validation
9. GridSearchCV Hyperparameter Tuning
10. Test Set Evaluation & Error Analysis
11. Model Interpretability (Log-Odds & Odds Ratios)
12. Inference Demonstration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

sns.set_theme(style='whitegrid')

### Step 1 & 2: Load Dataset and Inspect Missing Values

In [ ]:
data_path = '../backend/data/product_returns.csv'
df = pd.read_csv(data_path)
print(f'Total observations: {len(df)}, Features: {df.shape[1]}')

# Check duplicates
dups = df.duplicated().sum()
print(f'Duplicate records detected: {dups}')
df = df.drop_duplicates().reset_index(drop=True)

# Check missing values
print('\nMissing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

print('\nClass Balance:')
print(df['Return_Status'].value_counts(normalize=True))

### Step 3: Exploratory Data Analysis (Key Visualizations)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Return Rate by Category
cat_returns = df.groupby('Product_Category')['Return_Status'].mean().sort_values(ascending=False)
sns.barplot(x=cat_returns.values * 100, y=cat_returns.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Return Rate by Product Category (%)')
axes[0].set_xlabel('Return Rate (%)')

# 2. Delay vs Return
df['Delay'] = df['Delivery_Time'] - df['Expected_Delivery_Time']
sns.boxplot(x='Return_Status', y='Delay', data=df, ax=axes[1], palette=['#2563EB', '#DC2626'])
axes[1].set_title('Delivery Delay (Actual - Expected) by Return Status')
axes[1].set_xticklabels(['Retained (0)', 'Returned (1)'])

plt.tight_layout()
plt.show()

### Step 4 & 5: Feature Engineering & Stratified Train-Test Split
Crucial rule: Stratify on target to maintain class proportions, and split before fitting any transformers to ensure zero leakage.

In [ ]:
# Separate Features and Target
X = df.drop(columns=['Return_Status', 'Order_ID', 'Delay'], errors='ignore')
y = df['Return_Status'].values

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

### Step 6: Construct Zero-Leakage Preprocessing Pipeline

In [ ]:
import sys
sys.path.append('..')
from backend.ml.preprocessing import create_preprocessor

preprocessor = create_preprocessor()
log_reg = LogisticRegression(max_iter=1000, random_state=42)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', log_reg)
])

### Step 7 & 8: Baseline Comparison & 5-Fold Stratified Cross-Validation

In [ ]:
# Baseline Dummy Classifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print('Baseline Dummy Accuracy:', accuracy_score(y_test, dummy.predict(X_test)))

# 5-Fold Stratified CV on Training Set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])

print(f'5-Fold Mean Accuracy: {cv_results["test_accuracy"].mean():.4f}')
print(f'5-Fold Mean F1:       {cv_results["test_f1"].mean():.4f}')
print(f'5-Fold Mean ROC-AUC:  {cv_results["test_roc_auc"].mean():.4f}')

### Step 9: GridSearchCV Hyperparameter Tuning

In [ ]:
param_grid = {
    'classifier__C': [0.05, 0.1, 0.5, 1.0, 5.0],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs']
}

grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best Parameters:', grid.best_params_)
print(f'Best CV ROC-AUC: {grid.best_score_:.4f}')
best_model = grid.best_estimator_

### Step 10: Final Evaluation on Untouched Test Set

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('=== TEST SET EVALUATION ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Retained (0)', 'Predicted Returned (1)'],
            yticklabels=['Actual Retained (0)', 'Actual Returned (1)'])
plt.title('Confusion Matrix on Test Set')
plt.show()

### Step 11: Feature Importance (Logistic Regression Coefficients)

In [ ]:
from backend.ml.train import get_feature_names_from_preprocessor

feature_names = get_feature_names_from_preprocessor(best_model.named_steps['preprocessor'], X_train)
coefs = best_model.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs,
    'Odds_Ratio': np.exp(coefs),
    'Abs_Impact': np.abs(coefs)
}).sort_values(by='Abs_Impact', ascending=False)

plt.figure(figsize=(10, 6))
top_10 = coef_df.head(10)
sns.barplot(x='Coefficient', y='Feature', data=top_10, palette=['#DC2626' if c > 0 else '#16A34A' for c in top_10['Coefficient']])
plt.title('Top 10 Influential Features (Standardized Coefficients)')
plt.xlabel('Log-Odds Coefficient (β)')
plt.show()